# Inverted Pendulum: System Analysis

Key Python commands used in this tutorial are:
[`control.TransferFunction`](https://python-control.readthedocs.io/en/latest/generated/control.TransferFunction.html),
[`control.StateSpace`](https://python-control.readthedocs.io/en/latest/generated/control.StateSpace.html),
[`control.impulse_response`](https://python-control.readthedocs.io/en/latest/generated/control.impulse_response.html),
[`control.forced_response`](https://python-control.readthedocs.io/en/latest/generated/control.forced_response.html)

From the main problem, we derived the open-loop transfer functions of the inverted pendulum system as the following.

$$P_{pend}(s) = \frac{\Phi(s)}{U(s)}=\frac{\frac{ml}{q}s}{s^3+\frac{b(I+ml^2)}{q}s^2-\frac{(M+m)mgl}{q}s-\frac{bmgl}{q}} \qquad [ \frac{rad}{N}]$$

$$P_{cart}(s) =  \frac{X(s)}{U(s)} = \frac{ \frac{ (I+ml^2)s^2 - gml } {q}}{s^4+\frac{b(I+ml^2)}{q}s^3-\frac{(M+m)mgl}{q}s^2-\frac{bmgl}{q}s} \qquad [ \frac{m}{N}] $$

where

$$q = (M + m)(I + ml^2) - (ml)^2$$

Recall that the above two transfer functions are valid only for small values of the angle $\phi$ where $\phi$ is the deviation of the pendulum from the vertically upward position. Also, the absolute pendulum angle $\theta$ is equal to $\pi + \phi$.

For the original problem setup and the derivation of the above transfer functions, please refer to the [Inverted Pendulum: System Modeling](InvertedPendulum_SystemModeling.ipynb) page.

Considering the response of the pendulum to a 1-Nsec impulse applied to the cart, the design requirements for the pendulum are:

* Settling time for $\theta$ of less than 5 seconds
* Pendulum angle $\theta$ never more than 0.05 radians from the vertical

Additionally, the requirements for the response of the system to a 0.2-meter step command in cart position are:

* Settling time for $x$ and $\theta$ of less than 5 seconds
* Rise time for $x$ of less than 0.5 seconds
* Pendulum angle $\theta$ never more than 20 degrees (0.35 radians) from the vertical



## Open-loop impulse response

We will begin by looking at the open-loop response of the inverted pendulum system. Create a new code cell and type in the following commands to create the system model (refer to the main problem for the details of getting these commands).



In [ ]:
import control
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

sns.set(
    rc={
        "axes.labelsize": 8,
        "axes.titlesize": 8,
        "figure.figsize": (4 * 1.618, 4),
        "figure.dpi": 200,
    }
)

M = 0.5
m = 0.2
b = 0.1
I = 0.006
g = 9.8
l = 0.3
q = (M + m) * (I + m * l**2) - (m * l) ** 2
s = control.TransferFunction.s

P_cart = (((I + m * l**2) / q) * s**2 - (m * g * l / q)) / (
    s**4 + (b * (I + m * l**2)) * s**3 / q - ((M + m) * m * g * l) * s**2 / q - b * m * g * l * s / q
)

P_pend = (m * l * s / q) / (s**3 + (b * (I + m * l**2)) * s**2 / q - ((M + m) * m * g * l) * s / q - b * m * g * l / q)

We can now examine the open-loop impulse response of the system. Specifically, we will examine how the system responds to an impulsive force applied to the cart employing the Python command `control.impulse_response`. Add the following commands onto the end of your code cell and run it to get the associated plot shown below.



In [ ]:
t = np.arange(0, 1, 0.01)
T_cart, yout_cart = control.impulse_response(P_cart, T=t)
T_pend, yout_pend = control.impulse_response(P_pend, T=t)

plt.figure(figsize=(10, 6))
plt.subplot(2, 1, 1)
plt.plot(T_cart, yout_cart)
plt.title("Open-Loop Impulse Response - Cart Position")
plt.xlabel("Time (s)")
plt.ylabel("Cart Position (m)")
plt.grid("on")

plt.subplot(2, 1, 2)
plt.plot(T_pend, yout_pend)
plt.title("Open-Loop Impulse Response - Pendulum Angle")
plt.xlabel("Time (s)")
plt.ylabel("Pendulum Angle (rad)")
plt.grid("on")

plt.tight_layout()
plt.show()

As you can see from the plot, the system response is entirely unsatisfactory. In fact, it is not stable in open loop. Although the pendulum's position is shown to increase past 100 radians (15 revolutions), the model is only valid for small $\phi$. You can also see that the cart's position moves infinitely far to the right, though there is no requirement on cart position for an impulsive force input.

The poles of a system can also tell us about its time response. Since our system has two outputs and one input, it is described by two transfer functions. In general, all transfer functions from each input to each output of a multi-input, multi-output (MIMO) system will have the same poles (but different zeros) unless there are pole-zero cancellations. We will specifically examine the poles and zeros of the system using Python Control Systems Library functions.



The zeros and poles of the system where the pendulum position is the output are found as shown below:



In [ ]:
zeros_pend, poles_pend = control.pzmap(P_pend, plot=False)
print("Zeros of P_pend:", zeros_pend)
print("Poles of P_pend:", poles_pend)

Likewise, the zeros and poles of the system where the cart position is the output are found as follows:



In [ ]:
zeros_cart, poles_cart = control.pzmap(P_cart, plot=False)
print("Zeros of P_cart:", zeros_cart)
print("Poles of P_cart:", poles_cart)

As predicted, the poles for both transfer functions are identical. The pole with positive real part indicates that the system is unstable since the pole has positive real part. In other words, the pole is in the right half of the complex s-plane. This agrees with what we observed above.



## Open-loop step response

Since the system has a pole with positive real part its response to a step input will also grow unbounded. We will verify this using the `control.forced_response` command which can be employed to simulate the response of LTI models to arbitrary inputs. In this case, a 1-Newton step input will be used. Adding the following code to your code cell and running it will generate the plot shown below.



In [ ]:
t = np.arange(0, 10, 0.05)
u = np.ones_like(t)
T_cart, yout_cart, _ = control.forced_response(P_cart, T=t, U=u)
T_pend, yout_pend, _ = control.forced_response(P_pend, T=t, U=u)

plt.figure(figsize=(10, 6))
plt.plot(T_cart, yout_cart, label="x (cart position)")
plt.plot(T_pend, yout_pend, label="phi (pendulum angle)")
plt.title("Open-Loop Step Response")
plt.xlabel("Time (s)")
plt.ylabel("Response")
plt.axis([0, 3, 0, 50])
plt.legend()
plt.grid("on")
plt.show()

The above results confirm our expectation that the system's response to a step input is unstable.

It is apparent from the analysis above that some sort of control will need to be designed to improve the response of the system. Four example controllers are included with these tutorials: PID, root locus, frequency response, and state space. You may select a choice from the menu to the left for further details.

**Note**: The solutions shown in the PID, root locus, and frequency response examples may not yield a workable controller for the inverted pendulum problem. As stated previously, when we treat the inverted pendulum as a single-input, single-output system, we ignore the position of the cart, $x$. Where possible in these examples, we will show what happens to the cart's position when a controller is implemented on the system.

